In [1]:
#!/usr/bin/env python3
"""
直接生成pollutant_emission.bin - (30, 80, 180) float32格式
"""
import numpy as np
import pandas as pd

print("=== 生成pollutant_emission.bin ===")

# 读取河流数据
df = pd.read_csv('river_emissions.csv')
print(f"读取到 {len(df)} 条河流数据")

# 创建网格坐标
nx, ny = 180, 80
lon = np.linspace(-180, 180, nx)
lat = np.linspace(-80, 80, ny)

# 插值到网格 - 直接生成 (180, 80) 格式
emission_grid = np.zeros((nx, ny))  # (180, 80)

for _, river in df.iterrows():
    # 找到最近的网格点
    lon_idx = np.argmin(np.abs(lon - river['lon']))
    lat_idx = np.argmin(np.abs(lat - river['lat']))
    
    # 使用 Mid 值作为表面通量 (mol/m2/s)
    surface_flux = river['Mid'] / (365.25 * 24 * 3600)  # mol/m2/s
    
    # 累加到网格点
    emission_grid[lon_idx, lat_idx] += surface_flux

print(f"基础排放网格形状: {emission_grid.shape}")
print(f"基础排放范围: {emission_grid.min():.6e} 到 {emission_grid.max():.6e}")

# 生成30天的数据
daily_data = []
for day in range(30):
    # 添加简单的日变化
    daily_factor = 1.0 + 0.1 * np.sin(2 * np.pi * day / 7)
    daily_emission = emission_grid * daily_factor
    daily_emission = np.maximum(daily_emission, 0.0)
    daily_data.append(daily_emission)

daily_data = np.array(daily_data, dtype=np.float32)
print(f"每日数据形状: {daily_data.shape}")
print(f"每日数据类型: {daily_data.dtype}")

# 保存为二进制文件 - 按记录保存
filename = 'pollutant_emission.bin'
print(f"保存数据到 {filename}")

with open(filename, 'wb') as f:
    for record in range(daily_data.shape[0]):  # 30个时间记录
        # 直接使用 (180, 80) 格式，无需转置
        record_data = daily_data[record, :, :]  # (180, 80)
        # 转换为 Fortran 顺序 (列优先) 并保存
        record_data_fortran = np.asfortranarray(record_data, dtype=np.float32)
        record_data_fortran.tofile(f)

print(f"✅ 保存完成: {filename}")
print(f"数据形状: {daily_data.shape}")
print(f"数据类型: {daily_data.dtype}")
print(f"数据范围: {daily_data.min():.6e} 到 {daily_data.max():.6e}")
print(f"保存格式: 30个记录，每个记录 (180, 80) 的2D数组")

# 验证保存的文件
import os
file_size = os.path.getsize(filename)
expected_size = 30 * 180 * 80 * 4  # 30记录 * 180*80 * 4字节(float32)
print(f"文件大小: {file_size} 字节")
print(f"期望大小: {expected_size} 字节")

if file_size == expected_size:
    print("✅ 文件大小正确")
else:
    print("❌ 文件大小不正确")

# 验证数据格式
print("\n=== 验证数据格式 ===")
with open(filename, 'rb') as f:
    data = np.fromfile(f, dtype=np.float32)

print(f"读取数据形状: {data.shape}")
print(f"读取数据类型: {data.dtype}")
print(f"数据范围: {data.min():.6e} 到 {data.max():.6e}")

# 重塑数据为3D数组
data_3d = data.reshape(30, 180, 80)
print(f"重塑后形状: {data_3d.shape}")

# 检查每个记录
print(f"\n各记录统计:")
for i in range(min(3, 30)):  # 只显示前3个记录
    record_data = data_3d[i, :, :]
    print(f"  记录 {i+1}: 范围 {record_data.min():.6e} 到 {record_data.max():.6e}, 非零点数 {np.count_nonzero(record_data)}")

print("\n🎉 完成！数据文件已生成：pollutant_emission.bin")
print("格式：30个记录，每个记录 (180, 80) 的2D数组，float32精度")
print("数据形状：(30, 180, 80) - 直接匹配MITgcm期望格式")


=== 生成pollutant_emission.bin ===
读取到 1048 条河流数据
基础排放网格形状: (180, 80)
基础排放范围: 0.000000e+00 到 4.751527e-04
每日数据形状: (30, 180, 80)
每日数据类型: float32
保存数据到 pollutant_emission.bin
✅ 保存完成: pollutant_emission.bin
数据形状: (30, 180, 80)
数据类型: float32
数据范围: 0.000000e+00 到 5.214767e-04
保存格式: 30个记录，每个记录 (180, 80) 的2D数组
文件大小: 1728000 字节
期望大小: 1728000 字节
✅ 文件大小正确

=== 验证数据格式 ===
读取数据形状: (432000,)
读取数据类型: float32
数据范围: 0.000000e+00 到 5.214767e-04
重塑后形状: (30, 180, 80)

各记录统计:
  记录 1: 范围 0.000000e+00 到 4.751527e-04, 非零点数 622
  记录 2: 范围 0.000000e+00 到 5.123017e-04, 非零点数 622
  记录 3: 范围 0.000000e+00 到 5.214767e-04, 非零点数 622

🎉 完成！数据文件已生成：pollutant_emission.bin
格式：30个记录，每个记录 (180, 80) 的2D数组，float32精度
数据形状：(30, 180, 80) - 直接匹配MITgcm期望格式


In [2]:
#!/usr/bin/env python3
"""
测试pollutant_emission.bin数据文件
"""
import numpy as np

print("=== 测试pollutant_emission.bin数据文件 ===")

# 读取数据文件
data = np.fromfile('input/pollutant_emission.bin', dtype=np.float32)
print(f"原始数据形状: {data.shape}")
print(f"原始数据范围: {data.min():.6e} 到 {data.max():.6e}")
print(f"非零点数: {np.count_nonzero(data)}")

# 重塑为3D数组
data_3d = data.reshape(30, 180, 80)
print(f"重塑后形状: {data_3d.shape}")

# 检查每个记录
print(f"\n各记录统计:")
for i in range(min(5, 30)):  # 只显示前5个记录
    record_data = data_3d[i, :, :]
    print(f"  记录 {i+1}: 范围 {record_data.min():.6e} 到 {record_data.max():.6e}, 非零点数 {np.count_nonzero(record_data)}")

# 检查是否有异常值
print(f"\n异常值检查:")
print(f"是否有NaN: {np.isnan(data).any()}")
print(f"是否有Inf: {np.isinf(data).any()}")
print(f"是否有负值: {(data < 0).any()}")

# 检查最大值的位置
max_idx = np.argmax(data)
max_record = max_idx // (180 * 80)
max_pos = max_idx % (180 * 80)
max_i = max_pos // 80
max_j = max_pos % 80
print(f"最大值位置: 记录{max_record+1}, 位置({max_i}, {max_j}), 值: {data[max_idx]:.6e}")

# 检查数据文件大小
import os
file_size = os.path.getsize('input/pollutant_emission.bin')
expected_size = 30 * 180 * 80 * 4
print(f"\n文件大小: {file_size} 字节")
print(f"期望大小: {expected_size} 字节")
print(f"大小匹配: {file_size == expected_size}")


=== 测试pollutant_emission.bin数据文件 ===
原始数据形状: (432000,)
原始数据范围: 0.000000e+00 到 5.214767e-04
非零点数: 18660
重塑后形状: (30, 180, 80)

各记录统计:
  记录 1: 范围 0.000000e+00 到 4.751527e-04, 非零点数 622
  记录 2: 范围 0.000000e+00 到 5.123017e-04, 非零点数 622
  记录 3: 范围 0.000000e+00 到 5.214767e-04, 非零点数 622
  记录 4: 范围 0.000000e+00 到 4.957689e-04, 非零点数 622
  记录 5: 范围 0.000000e+00 到 4.545366e-04, 非零点数 622

异常值检查:
是否有NaN: False
是否有Inf: False
是否有负值: False
最大值位置: 记录3, 位置(150, 55), 值: 5.214767e-04

文件大小: 1728000 字节
期望大小: 1728000 字节
大小匹配: True
